<div class="alert alert-info" markdown="1">

#### Homework 9, Problem 7 Supplemental Notebook

# QR Factorization, Orthogonal Matrices, and Least Squares

### Math 124: Vectors, Matrices, and Applications, Fall 2026

<small><a style="text-decoration: none" href="https://math124.org">math124.org</a></small>

</div>

### Instructions

Most homeworks and some labs will have a Jupyter Notebook, like this one, containing Python code that supplements our understanding of the relevant mathematical ideas of the week.

---

Open this notebook in Google Colab. Instructions on how to use Google Colab are at [math124.org/running-code](https://math124.org/running-code).

**Most importantly, make sure to click "Copy to Drive" and rename the notebook before proceeding!**

---

Together, the tasks in this notebook constitute **Homework 9, Problem 7, worth 12 points**. Most of the code is supplied: you will replace **three marked `None` values**, in Tasks 1, 3, and 5, and use the supplied cells to examine your results.

You won't need to submit this notebook anywhere. Follow the **In Your PDF** instructions below: include screenshots of your code and its output, and write the requested explanations by hand. Upload your completed Homework 9 PDF to Pensive.

Work from top to bottom. Unfinished cells display a reminder so that you can run the notebook before completing every task.

## Problem 7: Billy the waiter

---

Suppose Billy works as a waiter at Mani Osteria. After serving 244 tables, he has recorded the total bill and the tip from each table. He wants to use these records to predict the tip he might receive from a future table.

Each row in the data describes one table. The column `total_bill` is the cost of the meal before the tip, and `tip` is the amount left for Billy. Both are measured in dollars.

Can we find a line that uses the total bill to predict the tip? The data points do not all lie on one line, so we cannot make every prediction exactly correct. We will use least squares to choose the coefficients, then investigate what the predictions tell us.

Run the next cell. The data are included in this notebook, so you do not need to download or upload another file.

In [ ]:
#@title Load the packages and Billy's tips data { display-mode: "form" }
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from io import StringIO
from IPython.display import display, clear_output

np.set_printoptions(precision=6, suppress=True)
pio.templates["math124"] = go.layout.Template(layout=dict(
    font=dict(family="Palatino, Palatino Linotype, Book Antiqua, serif", size=17),
    paper_bgcolor="white", plot_bgcolor="white",
    colorway=["#1D4ED8", "#B04F09", "#A21C58"],
    xaxis=dict(showgrid=True, gridcolor="#E2E8F0", zeroline=False),
    yaxis=dict(showgrid=True, gridcolor="#E2E8F0", zeroline=False)))
pio.templates.default = "plotly_white+math124"

# The data are included here so this notebook runs without another file.
_TIPS_CSV = """total_bill,tip,sex,smoker,day,time,table_size
16.99,1.01,Female,No,Sun,Dinner,2
10.34,1.66,Male,No,Sun,Dinner,3
21.01,3.5,Male,No,Sun,Dinner,3
23.68,3.31,Male,No,Sun,Dinner,2
24.59,3.61,Female,No,Sun,Dinner,4
25.29,4.71,Male,No,Sun,Dinner,4
8.77,2.0,Male,No,Sun,Dinner,2
26.88,3.12,Male,No,Sun,Dinner,4
15.04,1.96,Male,No,Sun,Dinner,2
14.78,3.23,Male,No,Sun,Dinner,2
10.27,1.71,Male,No,Sun,Dinner,2
35.26,5.0,Female,No,Sun,Dinner,4
15.42,1.57,Male,No,Sun,Dinner,2
18.43,3.0,Male,No,Sun,Dinner,4
14.83,3.02,Female,No,Sun,Dinner,2
21.58,3.92,Male,No,Sun,Dinner,2
10.33,1.67,Female,No,Sun,Dinner,3
16.29,3.71,Male,No,Sun,Dinner,3
16.97,3.5,Female,No,Sun,Dinner,3
20.65,3.35,Male,No,Sat,Dinner,3
17.92,4.08,Male,No,Sat,Dinner,2
20.29,2.75,Female,No,Sat,Dinner,2
15.77,2.23,Female,No,Sat,Dinner,2
39.42,7.58,Male,No,Sat,Dinner,4
19.82,3.18,Male,No,Sat,Dinner,2
17.81,2.34,Male,No,Sat,Dinner,4
13.37,2.0,Male,No,Sat,Dinner,2
12.69,2.0,Male,No,Sat,Dinner,2
21.7,4.3,Male,No,Sat,Dinner,2
19.65,3.0,Female,No,Sat,Dinner,2
9.55,1.45,Male,No,Sat,Dinner,2
18.35,2.5,Male,No,Sat,Dinner,4
15.06,3.0,Female,No,Sat,Dinner,2
20.69,2.45,Female,No,Sat,Dinner,4
17.78,3.27,Male,No,Sat,Dinner,2
24.06,3.6,Male,No,Sat,Dinner,3
16.31,2.0,Male,No,Sat,Dinner,3
16.93,3.07,Female,No,Sat,Dinner,3
18.69,2.31,Male,No,Sat,Dinner,3
31.27,5.0,Male,No,Sat,Dinner,3
16.04,2.24,Male,No,Sat,Dinner,3
17.46,2.54,Male,No,Sun,Dinner,2
13.94,3.06,Male,No,Sun,Dinner,2
9.68,1.32,Male,No,Sun,Dinner,2
30.4,5.6,Male,No,Sun,Dinner,4
18.29,3.0,Male,No,Sun,Dinner,2
22.23,5.0,Male,No,Sun,Dinner,2
32.4,6.0,Male,No,Sun,Dinner,4
28.55,2.05,Male,No,Sun,Dinner,3
18.04,3.0,Male,No,Sun,Dinner,2
12.54,2.5,Male,No,Sun,Dinner,2
10.29,2.6,Female,No,Sun,Dinner,2
34.81,5.2,Female,No,Sun,Dinner,4
9.94,1.56,Male,No,Sun,Dinner,2
25.56,4.34,Male,No,Sun,Dinner,4
19.49,3.51,Male,No,Sun,Dinner,2
38.01,3.0,Male,Yes,Sat,Dinner,4
26.41,1.5,Female,No,Sat,Dinner,2
11.24,1.76,Male,Yes,Sat,Dinner,2
48.27,6.73,Male,No,Sat,Dinner,4
20.29,3.21,Male,Yes,Sat,Dinner,2
13.81,2.0,Male,Yes,Sat,Dinner,2
11.02,1.98,Male,Yes,Sat,Dinner,2
18.29,3.76,Male,Yes,Sat,Dinner,4
17.59,2.64,Male,No,Sat,Dinner,3
20.08,3.15,Male,No,Sat,Dinner,3
16.45,2.47,Female,No,Sat,Dinner,2
3.07,1.0,Female,Yes,Sat,Dinner,1
20.23,2.01,Male,No,Sat,Dinner,2
15.01,2.09,Male,Yes,Sat,Dinner,2
12.02,1.97,Male,No,Sat,Dinner,2
17.07,3.0,Female,No,Sat,Dinner,3
26.86,3.14,Female,Yes,Sat,Dinner,2
25.28,5.0,Female,Yes,Sat,Dinner,2
14.73,2.2,Female,No,Sat,Dinner,2
10.51,1.25,Male,No,Sat,Dinner,2
17.92,3.08,Male,Yes,Sat,Dinner,2
27.2,4.0,Male,No,Thur,Lunch,4
22.76,3.0,Male,No,Thur,Lunch,2
17.29,2.71,Male,No,Thur,Lunch,2
19.44,3.0,Male,Yes,Thur,Lunch,2
16.66,3.4,Male,No,Thur,Lunch,2
10.07,1.83,Female,No,Thur,Lunch,1
32.68,5.0,Male,Yes,Thur,Lunch,2
15.98,2.03,Male,No,Thur,Lunch,2
34.83,5.17,Female,No,Thur,Lunch,4
13.03,2.0,Male,No,Thur,Lunch,2
18.28,4.0,Male,No,Thur,Lunch,2
24.71,5.85,Male,No,Thur,Lunch,2
21.16,3.0,Male,No,Thur,Lunch,2
28.97,3.0,Male,Yes,Thur,Dinner,2
22.49,3.5,Male,No,Thur,Dinner,2
5.75,1.0,Female,Yes,Thur,Dinner,2
16.32,4.3,Female,Yes,Thur,Dinner,2
22.75,3.25,Female,No,Thur,Dinner,2
40.17,4.73,Male,Yes,Thur,Dinner,4
27.28,4.0,Male,Yes,Thur,Dinner,2
12.03,1.5,Male,Yes,Thur,Dinner,2
21.01,3.0,Male,Yes,Thur,Dinner,2
12.46,1.5,Male,No,Thur,Dinner,2
11.35,2.5,Female,Yes,Thur,Dinner,2
15.38,3.0,Female,Yes,Thur,Dinner,2
44.3,2.5,Female,Yes,Sat,Dinner,3
22.42,3.48,Female,Yes,Sat,Dinner,2
20.92,4.08,Female,No,Sat,Dinner,2
15.36,1.64,Male,Yes,Sat,Dinner,2
20.49,4.06,Male,Yes,Sat,Dinner,2
25.21,4.29,Male,Yes,Sat,Dinner,2
18.24,3.76,Male,No,Sat,Dinner,2
14.31,4.0,Female,Yes,Sat,Dinner,2
14.0,3.0,Male,No,Sat,Dinner,2
7.25,1.0,Female,No,Sat,Dinner,1
38.07,4.0,Male,No,Sun,Dinner,3
23.95,2.55,Male,No,Sun,Dinner,2
25.71,4.0,Female,No,Sun,Dinner,3
17.31,3.5,Female,No,Sun,Dinner,2
29.93,5.07,Male,No,Sun,Dinner,4
10.65,1.5,Female,No,Thur,Lunch,2
12.43,1.8,Female,No,Thur,Lunch,2
24.08,2.92,Female,No,Thur,Lunch,4
11.69,2.31,Male,No,Thur,Lunch,2
13.42,1.68,Female,No,Thur,Lunch,2
14.26,2.5,Male,No,Thur,Lunch,2
15.95,2.0,Male,No,Thur,Lunch,2
12.48,2.52,Female,No,Thur,Lunch,2
29.8,4.2,Female,No,Thur,Lunch,6
8.52,1.48,Male,No,Thur,Lunch,2
14.52,2.0,Female,No,Thur,Lunch,2
11.38,2.0,Female,No,Thur,Lunch,2
22.82,2.18,Male,No,Thur,Lunch,3
19.08,1.5,Male,No,Thur,Lunch,2
20.27,2.83,Female,No,Thur,Lunch,2
11.17,1.5,Female,No,Thur,Lunch,2
12.26,2.0,Female,No,Thur,Lunch,2
18.26,3.25,Female,No,Thur,Lunch,2
8.51,1.25,Female,No,Thur,Lunch,2
10.33,2.0,Female,No,Thur,Lunch,2
14.15,2.0,Female,No,Thur,Lunch,2
16.0,2.0,Male,Yes,Thur,Lunch,2
13.16,2.75,Female,No,Thur,Lunch,2
17.47,3.5,Female,No,Thur,Lunch,2
34.3,6.7,Male,No,Thur,Lunch,6
41.19,5.0,Male,No,Thur,Lunch,5
27.05,5.0,Female,No,Thur,Lunch,6
16.43,2.3,Female,No,Thur,Lunch,2
8.35,1.5,Female,No,Thur,Lunch,2
18.64,1.36,Female,No,Thur,Lunch,3
11.87,1.63,Female,No,Thur,Lunch,2
9.78,1.73,Male,No,Thur,Lunch,2
7.51,2.0,Male,No,Thur,Lunch,2
14.07,2.5,Male,No,Sun,Dinner,2
13.13,2.0,Male,No,Sun,Dinner,2
17.26,2.74,Male,No,Sun,Dinner,3
24.55,2.0,Male,No,Sun,Dinner,4
19.77,2.0,Male,No,Sun,Dinner,4
29.85,5.14,Female,No,Sun,Dinner,5
48.17,5.0,Male,No,Sun,Dinner,6
25.0,3.75,Female,No,Sun,Dinner,4
13.39,2.61,Female,No,Sun,Dinner,2
16.49,2.0,Male,No,Sun,Dinner,4
21.5,3.5,Male,No,Sun,Dinner,4
12.66,2.5,Male,No,Sun,Dinner,2
16.21,2.0,Female,No,Sun,Dinner,3
13.81,2.0,Male,No,Sun,Dinner,2
17.51,3.0,Female,Yes,Sun,Dinner,2
24.52,3.48,Male,No,Sun,Dinner,3
20.76,2.24,Male,No,Sun,Dinner,2
31.71,4.5,Male,No,Sun,Dinner,4
10.59,1.61,Female,Yes,Sat,Dinner,2
10.63,2.0,Female,Yes,Sat,Dinner,2
50.81,10.0,Male,Yes,Sat,Dinner,3
15.81,3.16,Male,Yes,Sat,Dinner,2
7.25,5.15,Male,Yes,Sun,Dinner,2
31.85,3.18,Male,Yes,Sun,Dinner,2
16.82,4.0,Male,Yes,Sun,Dinner,2
32.9,3.11,Male,Yes,Sun,Dinner,2
17.89,2.0,Male,Yes,Sun,Dinner,2
14.48,2.0,Male,Yes,Sun,Dinner,2
9.6,4.0,Female,Yes,Sun,Dinner,2
34.63,3.55,Male,Yes,Sun,Dinner,2
34.65,3.68,Male,Yes,Sun,Dinner,4
23.33,5.65,Male,Yes,Sun,Dinner,2
45.35,3.5,Male,Yes,Sun,Dinner,3
23.17,6.5,Male,Yes,Sun,Dinner,4
40.55,3.0,Male,Yes,Sun,Dinner,2
20.69,5.0,Male,No,Sun,Dinner,5
20.9,3.5,Female,Yes,Sun,Dinner,3
30.46,2.0,Male,Yes,Sun,Dinner,5
18.15,3.5,Female,Yes,Sun,Dinner,3
23.1,4.0,Male,Yes,Sun,Dinner,3
15.69,1.5,Male,Yes,Sun,Dinner,2
19.81,4.19,Female,Yes,Thur,Lunch,2
28.44,2.56,Male,Yes,Thur,Lunch,2
15.48,2.02,Male,Yes,Thur,Lunch,2
16.58,4.0,Male,Yes,Thur,Lunch,2
7.56,1.44,Male,No,Thur,Lunch,2
10.34,2.0,Male,Yes,Thur,Lunch,2
43.11,5.0,Female,Yes,Thur,Lunch,4
13.0,2.0,Female,Yes,Thur,Lunch,2
13.51,2.0,Male,Yes,Thur,Lunch,2
18.71,4.0,Male,Yes,Thur,Lunch,3
12.74,2.01,Female,Yes,Thur,Lunch,2
13.0,2.0,Female,Yes,Thur,Lunch,2
16.4,2.5,Female,Yes,Thur,Lunch,2
20.53,4.0,Male,Yes,Thur,Lunch,4
16.47,3.23,Female,Yes,Thur,Lunch,3
26.59,3.41,Male,Yes,Sat,Dinner,3
38.73,3.0,Male,Yes,Sat,Dinner,4
24.27,2.03,Male,Yes,Sat,Dinner,2
12.76,2.23,Female,Yes,Sat,Dinner,2
30.06,2.0,Male,Yes,Sat,Dinner,3
25.89,5.16,Male,Yes,Sat,Dinner,4
48.33,9.0,Male,No,Sat,Dinner,4
13.27,2.5,Female,Yes,Sat,Dinner,2
28.17,6.5,Female,Yes,Sat,Dinner,3
12.9,1.1,Female,Yes,Sat,Dinner,2
28.15,3.0,Male,Yes,Sat,Dinner,5
11.59,1.5,Male,Yes,Sat,Dinner,2
7.74,1.44,Male,Yes,Sat,Dinner,2
30.14,3.09,Female,Yes,Sat,Dinner,4
12.16,2.2,Male,Yes,Thur,Lunch,2
13.42,3.48,Female,Yes,Thur,Lunch,2
8.58,1.92,Male,Yes,Thur,Lunch,1
15.98,3.0,Female,No,Thur,Lunch,3
13.42,1.58,Male,Yes,Thur,Lunch,2
16.27,2.5,Female,Yes,Thur,Lunch,2
10.09,2.0,Female,Yes,Thur,Lunch,2
20.45,3.0,Male,No,Sat,Dinner,4
13.28,2.72,Male,No,Sat,Dinner,2
22.12,2.88,Female,Yes,Sat,Dinner,2
24.01,2.0,Male,Yes,Sat,Dinner,4
15.69,3.0,Male,Yes,Sat,Dinner,3
11.61,3.39,Male,No,Sat,Dinner,2
10.77,1.47,Male,No,Sat,Dinner,2
15.53,3.0,Male,Yes,Sat,Dinner,2
10.07,1.25,Male,No,Sat,Dinner,2
12.6,1.0,Male,Yes,Sat,Dinner,2
32.83,1.17,Male,Yes,Sat,Dinner,2
35.83,4.67,Female,No,Sat,Dinner,3
29.03,5.92,Male,No,Sat,Dinner,3
27.18,2.0,Female,Yes,Sat,Dinner,2
22.67,2.0,Male,Yes,Sat,Dinner,2
17.82,1.75,Male,No,Sat,Dinner,2
18.78,3.0,Female,No,Thur,Dinner,2
"""
tips = pd.read_csv(StringIO(_TIPS_CSV))
print(f"Number of tables: {len(tips)}")
display(tips[["total_bill", "tip"]].head())

In [ ]:
# Run this cell to see the data.
fig = go.Figure(go.Scatter(
    x=tips["total_bill"], y=tips["tip"], mode="markers", name="Observed tips",
    marker=dict(size=8, color="#1D4ED8", opacity=0.75),
    hovertemplate="Bill: $%{x:.2f}<br>Tip: $%{y:.2f}<extra></extra>"))
fig.update_layout(title="Billy's tips", xaxis_title="Total bill (dollars)",
                  yaxis_title="Tip (dollars)", height=460,
                  margin=dict(l=65, r=25, t=65, b=60))
fig.show()

## The model and design matrix

---

As in the lecture worksheet, we will fit a line

$$y=c+mt.$$

Here, $t$ is the total bill in dollars, and $y$ is the predicted tip in dollars. The intercept is $c$ and the slope is $m$, so our coefficient vector is

$$\vec x=\begin{bmatrix}c\\m\end{bmatrix}.$$

For a table with bill $t_i$ and observed tip $b_i$, we would like $c+mt_i=b_i$. The **design matrix** $A$ puts these equations together: its first column contains ones and its second contains the bills. The vector $\vec b$ contains the observed tips:

$$
A=\begin{bmatrix}
1 & t_1\\
1 & t_2\\
\vdots & \vdots\\
1 & t_{244}
\end{bmatrix},\qquad
\vec b=\begin{bmatrix}
b_1\\b_2\\\vdots\\b_{244}
\end{bmatrix}.
$$

The product $A\vec x$ gives the predicted tip for every table, in the same order as $\vec b$. We cannot make $A\vec x=\vec b$ hold exactly for these data. Least squares chooses $c$ and $m$ to make $A\vec x$ as close to $\vec b$ as possible.

Run the supplied code below. You do not need to edit `create_design_matrix`. In Python, `A` is a two-dimensional array and `b` is a one-dimensional array. Use `@` for matrix multiplication, `.T` for transpose, and `np.dot(u, v)` for the dot product of two vectors.

In [ ]:
def create_design_matrix(data, columns):
    """Return a column of ones followed by the requested data columns."""
    features = data[columns].to_numpy(dtype=float)
    return np.column_stack((np.ones(len(data)), features))

A = create_design_matrix(tips, ["total_bill"])
b = tips["tip"].to_numpy(dtype=float)

print("A has shape", A.shape)
print("b has shape", b.shape)
print("First five rows of A:\n", A[:5])
print("First five entries of b:", b[:5])

## Task 1: Fit the line (3 points)

---

The least-squares coefficients $\vec x^*$ satisfy the normal equations

$$A^T A\vec x^*=A^T\vec b.$$

The columns of $A$ are linearly independent, so $A^TA$ is invertible. The NumPy function `np.linalg.solve(M, v)` finds the vector `z` satisfying `M @ z = v`.

Replace `None` in **TODO 1** with one call to `np.linalg.solve` that solves the normal equations. You should use `A` and `b` in your expression.

**In Your PDF:** Include a screenshot of the completed cell and its output. Write your fitted line in the form $y=c+mt$, rounding $c$ and $m$ to four decimal places. Keep full precision in your code.

In [ ]:
# TODO 1: Solve the normal equations for the two coefficients.
x_star = None

# Everything below is supplied.
if x_star is None:
    print("Complete TODO 1, then run this cell again.")
else:
    print("x_star =", x_star)
    print(f"c = {x_star[0]:.6f}")
    print(f"m = {x_star[1]:.6f}")
    print(f"y = {x_star[0]:.6f} + {x_star[1]:.6f} t")

## Task 2: Interpret the slope (1 point)

---

According to your fitted line $y=c+mt$, how much does the predicted tip increase when the bill increases by \$10?

**In Your PDF:** Give the increase in dollars and explain how you used $m$ to find it.

## Task 3: Predict one tip (2 points)

---

A new table has a total bill of \$38.50. To make a prediction, place a $1$ and the total bill in a vector:

$$\vec a_{\mathrm{new}}=\begin{bmatrix}1\\38.50\end{bmatrix}.$$

Its predicted tip is the dot product $\vec a_{\mathrm{new}}\cdot\vec x^*$.

Complete **TODO 2** in `predict_tip`. Its argument `total_bill` is the bill for one table. Use `np.dot` with the supplied vector `a_new` and the fitted coefficients `x_star` from Task 1. Then run the supplied call to make Billy's prediction.

**In Your PDF:** Include a screenshot of the completed function and its output. State the predicted tip for the \$38.50 bill, rounded to the nearest cent.

In [ ]:
def predict_tip(total_bill):
    """Predict the tip for one table using the fitted coefficients."""
    a_new = np.array([1.0, total_bill])
    # TODO 2: Take the dot product of a_new and x_star.
    prediction = None
    return prediction

# Everything below is supplied.
if x_star is None:
    print("Complete Task 1 first.")
else:
    predicted_tip = predict_tip(38.50)
    if predicted_tip is None:
        print("Complete TODO 2, then run this cell again.")
    else:
        print(f"Predicted tip for a $38.50 bill: ${predicted_tip:.6f}")

## Task 4: Assess a prediction (2 points)

---

Now suppose Billy serves a table with a total bill of \$3,000. The function will still return a number, but does that make it a useful prediction?

Run the supplied cell below. Compare \$3,000 with the bills Billy has recorded.

**In Your PDF:** Report the value of `predict_tip(3000)`, rounded to the nearest cent. Do you think this is a useful prediction of the tip Billy would receive? Explain using the range of bills in the data.

In [ ]:
# This code is supplied; run it after completing predict_tip.
print(f"Bills in the data range from ${A[:, 1].min():.2f} to ${A[:, 1].max():.2f}.")
if x_star is None:
    print("Complete Task 1 first.")
else:
    large_bill_tip = predict_tip(3000)
    if large_bill_tip is None:
        print("Complete TODO 2, then run this cell again.")
    else:
        print(f"Predicted tip for a $3,000 bill: ${large_bill_tip:.6f}")

## Task 5: Compute the mean squared error (2 points)

---

The vector $A\vec x$ gives all 244 predicted tips. Subtracting it from $\vec b$ gives the **error vector** $\vec e=\vec b-A\vec x$. Each entry is the observed tip minus the predicted tip for one table.

The **mean squared error (MSE)** is the average of the squared errors:

$$\operatorname{MSE}=\frac{e_1^2+e_2^2+\cdots+e_{244}^2}{244}.$$

Complete **TODO 3** in `compute_mse`. The error vector is already computed for you and stored in `errors`. For a NumPy array, `** 2` squares each entry and `np.mean` takes the average.

The supplied code will display the fitted line's MSE and the first five predictions and errors.

**In Your PDF:** Include a screenshot of your completed function and its output. Report the least-squares MSE to four decimal places.

In [ ]:
def compute_mse(A, b, x):
    """Return the mean squared error for the predictions A @ x."""
    errors = b - A @ x
    # TODO 3: Find the average of the squared entries of errors.
    mse = None
    return mse

# Everything below is supplied.
if x_star is None:
    print("Complete Task 1 first.")
else:
    fitted_mse = compute_mse(A, b, x_star)
    if fitted_mse is None:
        print("Complete TODO 3, then run this cell again.")
    else:
        predictions = A @ x_star
        errors = b - predictions
        print(f"Least-squares MSE: {fitted_mse:.6f} dollars squared")
        display(pd.DataFrame({
            "Total bill": tips["total_bill"].to_numpy()[:5],
            "Observed tip": b[:5],
            "Predicted tip": predictions[:5],
            "Error": errors[:5]
        }))

## Task 6: Compare another line (2 points)

---

Run the supplied cell below. Use the sliders for $c$ and $m$ to try a different line, and compare its MSE with the least-squares line. You can turn on the vertical error lines to see the difference between each observed tip and its prediction.

**In Your PDF:** Give $c$, $m$, and the MSE for one different line you tried. State which line has the smaller MSE.

In [ ]:
#@title Experiment with the intercept and slope { display-mode: "form" }
def show_line_explorer():
    if x_star is None or compute_mse(A, b, x_star) is None:
        print("Complete Tasks 1 and 5, then rerun this cell.")
        return

    intercept = widgets.FloatSlider(
        value=float(x_star[0]), min=-2, max=4, step=0.01,
        description="Intercept c", readout_format=".4f", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
    slope = widgets.FloatSlider(
        value=float(x_star[1]), min=0, max=0.30, step=0.002,
        description="Slope m", readout_format=".4f", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
    show_errors = widgets.Checkbox(value=False, description="Show vertical error lines")
    reset = widgets.Button(description="Reset to least squares", layout=widgets.Layout(width="200px"))
    output = widgets.Output()
    bill_grid = np.linspace(0, 55, 150)
    fitted_mse = compute_mse(A, b, x_star)

    def redraw(change=None):
        trial = np.array([intercept.value, slope.value])
        trial_mse = compute_mse(A, b, trial)
        plot = go.Figure()
        if show_errors.value:
            segments_bill, segments_tip = [], []
            for bill, observed, predicted in zip(A[:, 1], b, A @ trial):
                segments_bill.extend([bill, bill, None])
                segments_tip.extend([observed, predicted, None])
            plot.add_trace(go.Scatter(x=segments_bill, y=segments_tip, mode="lines",
                line=dict(color="rgba(162,28,88,0.35)", width=1.3),
                name="Trial errors", hoverinfo="skip"))
        plot.add_trace(go.Scatter(x=A[:, 1], y=b, mode="markers", name="Observed tips",
            marker=dict(size=8, color="#1D4ED8", opacity=0.70),
            hovertemplate="Bill: $%{x:.2f}<br>Tip: $%{y:.2f}<extra></extra>"))
        plot.add_trace(go.Scatter(x=bill_grid, y=x_star[0]+x_star[1]*bill_grid,
            mode="lines", name="Least-squares line",
            line=dict(color="#B04F09", width=4, dash="dash")))
        plot.add_trace(go.Scatter(x=bill_grid, y=trial[0]+trial[1]*bill_grid,
            mode="lines", name="Your trial line", line=dict(color="#A21C58", width=3)))
        plot.update_layout(title="Compare a trial line with the least-squares line",
            xaxis_title="Total bill (dollars)", yaxis_title="Tip (dollars)",
            xaxis_range=[0, 55], yaxis_range=[-2, 12], height=490,
            legend=dict(orientation="h", y=1.02, x=0),
            margin=dict(l=65, r=25, t=105, b=60))
        with output:
            clear_output(wait=True)
            print(f"Trial: intercept = {trial[0]:.6f}, slope = {trial[1]:.6f}, MSE = {trial_mse:.6f}")
            print(f"Least-squares MSE = {fitted_mse:.6f}")
            plot.show()

    def reset_to_fit(button):
        intercept.value = float(x_star[0])
        slope.value = float(x_star[1])
        redraw()

    for control in (intercept, slope, show_errors):
        control.observe(redraw, names="value")
    reset.on_click(reset_to_fit)
    display(widgets.VBox([intercept, slope, show_errors, reset, output]))
    redraw()

show_line_explorer()

## Submission checklist

---

Include the following in your Homework 9 PDF:

- **Task 1:** Your completed normal-equations code and its output, and your fitted line $y=c+mt$.
- **Task 2:** Your interpretation of the slope.
- **Task 3:** Your completed `predict_tip` function and the prediction for the \$38.50 bill.
- **Task 4:** The prediction for the \$3,000 bill and your explanation of whether it is useful.
- **Task 5:** Your completed `compute_mse` function and the least-squares MSE.
- **Task 6:** The values of $c$, $m$, and MSE for another line, and your comparison.

You do not need to screenshot the supplied setup code, dataset, or plot-building code. Keep full precision in your calculations; round only the values requested in your written answers.